# MarkovMPS walkthrough

This notebook answers one question: how does the markovmps package turn a first-passage or avalanche question into tensor-train linear algebra, and how is each number it produces checked? After completing it, a reader can build a model, call the solver an observable needs, and verify the answer against an independent reference.

In scope: the two committed problem families at small sizes, the biased walk between absorbing barriers and the Manna sandpile, with every engine the package carries. Out of scope: classifying regimes, fitting exponents, and producing publication figures, which are the reader's own analysis (the boundary is stated in [docs/notes.md](../docs/notes.md)); and large near-critical systems, whose cost is stated in the README's note on cost. The reader is assumed to know what a Markov process and a probability distribution are; no tensor-network background is assumed.

The notebook uses the Python package installed from [Python/](../Python/) (`pip install -e Python`), imports every function from it, and writes no files; the figures render inline.

In [1]:
import matplotlib.pyplot as plt
import numpy as np

from markovmps import (BiasedWalk, MannaSandpile, exact_avalanche_size_distribution,
                       exact_mfpt, exact_survival, kmc_avalanche_sizes, kmc_survival,
                       max_abs_deviation)
from markovmps.models.firstpassage import tt_mfpt, tt_survival
from markovmps.models.manna import tt_avalanche_size_distribution, tt_mean_avalanche_size
from markovmps.plot_utils import PLOT_DEFAULTS

## Parameters

Every configurable value lives in this one cell, so a reader can reproduce or vary any run without hunting through the notebook. The walk parameters match the committed `barrier` instance and the pile parameters match `manna_small`, so every number below can also be regenerated from the command line by name. The bond dimension `MAXDIM` is deliberately absent here: it trades accuracy against cost, so it is set by the convergence cell below, not assumed.

In [2]:
# -----------------------------------------------
# Parameters -- edit here
# -----------------------------------------------
SEED          = 1            # Monte Carlo seed; without it a sampled run is not reproducible
# biased walk (the committed barrier instance)
NBITS         = 4            # binary sites, so the walk lives on 2**NBITS positions
RATE_UP       = 1.3          # upward hop rate
RATE_DOWN     = 0.7          # downward hop rate
START         = 5            # starting position
TMAX          = 6.0          # longest time on the survival curve
TPOINTS       = 25           # times on the survival curve
# Manna sandpile (the committed manna_small instance)
SITES         = 4            # ring size
GRAINS        = 3            # grain count
SMAX          = 24           # largest avalanche size resolved by the inversion
SMAX_TAIL     = 800          # long exact run used only to bound the aliased tail mass
# engines and numerics
NSHOTS        = 4000         # Monte Carlo trajectories
CUTOFF        = 1e-10        # singular values below this are discarded
MAXDIM_LADDER = [4, 8, 16, 32]   # bond dimensions the convergence cell walks
CONV_TOL      = 1e-9         # the mean is converged when a ladder step moves it less than this

# palette: the package's own plotting constants, one colour per engine
COLOUR_CLOSED = PLOT_DEFAULTS["closed_form_colour"]
COLOUR_EXACT  = PLOT_DEFAULTS["exact_colour"]
COLOUR_KMC    = PLOT_DEFAULTS["kmc_colour"]
COLOUR_TT     = PLOT_DEFAULTS["tt_colour"]

## The biased walk between absorbing barriers

A walker hops on the integers 0 to $2^{n}-1$ with rate `RATE_UP` upward and `RATE_DOWN` downward, and the two end positions absorb it. Every observable of this model has a closed form, which is why it anchors the package: any construction error surfaces as a disagreement with a formula. The mean first-passage time from position $x$ is the gambler's ruin result,

$$\tau(x) = \frac{x - L\,h(x) - R\,(1-h(x))\,\cdot\,0}{\,r_\downarrow - r_\uparrow\,}\quad\text{(up to the boundary terms the package's docstring states)},$$

where $r_\uparrow$ is `RATE_UP`, $r_\downarrow$ is `RATE_DOWN`, $h(x)$ is the splitting probability of reaching the right barrier first, and the package evaluates the full expression as `walk.analytic_mfpt()`. A typical value at these parameters is of order 10 time units.

In [3]:
walk = BiasedWalk(NBITS, RATE_UP, RATE_DOWN, START)
closed_mfpt = walk.analytic_mfpt()
print(f"walk on {2**NBITS} positions, start {START}, closed-form MFPT {closed_mfpt:.12g}")

walk on 16 positions, start 5, closed-form MFPT 15.5372272557


### Smoke test

Before any figure, a property no correct implementation can violate. The dense exact engine builds the generator with no truncation, so its hitting time must match the closed form to solver precision; the tensor-train solve reports a relative residual beside its answer, and the answer must sit within a small multiple of the closed form. An assertion that fails here stops the notebook before it can produce a plausible wrong curve.

In [4]:
dense_mfpt = exact_mfpt(walk)
tt_value, tt_residual = tt_mfpt(walk, cutoff=CUTOFF)
assert abs(dense_mfpt - closed_mfpt) < 1e-9 * closed_mfpt, "dense engine disagrees with the closed form"
assert abs(tt_value - closed_mfpt) < 1e-6 * closed_mfpt, "tensor-train solve disagrees with the closed form"
print(f"closed form {closed_mfpt:.12g}")
print(f"dense       {dense_mfpt:.12g}")
print(f"tensor train {tt_value:.12g}   relative residual {tt_residual:.1e}")

closed form 15.5372272557
dense       15.5372272557
tensor train 15.5372272557   relative residual 8.7e-15


### The survival curve from every engine

The survival probability $S(t)$, the chance the walker has not yet been absorbed by time $t$, decays as a sum of exponentials whose rates are the transient spectrum. The closed spectral form is drawn as a line and the three engines as markers on top of it; the thing to look for is that all three sit on the line, with the Monte Carlo points scattering by their sampling error while the other two do not.

In [5]:
times = np.linspace(0.0, TMAX, TPOINTS)
surv_closed = walk.analytic_survival(times)
surv_exact = exact_survival(walk, times)
surv_kmc = kmc_survival(walk, times, NSHOTS, SEED)
surv_tt = tt_survival(walk, times, cutoff=CUTOFF)

fig, ax = plt.subplots(figsize=(6.0, 3.6))
ax.plot(times, surv_closed, "-", color=COLOUR_CLOSED, label="closed form")
ax.plot(times, surv_exact, "o", ms=4, color=COLOUR_EXACT, label="exact")
ax.plot(times, surv_kmc, "s", ms=4, color=COLOUR_KMC, label="kmc")
ax.plot(times, surv_tt, "^", ms=4, color=COLOUR_TT, label="tensor train")
ax.set_xlabel("time")
ax.set_ylabel("survival probability")
ax.set_yscale("log")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
print(f"largest tensor-train deviation from the closed form {max_abs_deviation(surv_tt, surv_closed):.1e}")

largest tensor-train deviation from the closed form 7.5e-09


## The Manna sandpile

`GRAINS` grains sit on a ring of `SITES` sites; a site holding 2 or more grains is active and topples, sending 2 grains to independently random neighbours, and the avalanche is the total number of topplings before no active site remains. The observable is the distribution of that size $S$, reached through the probability generating function

$$G(z) = \mathrm{E}\!\left[z^{S}\right],$$

where `tt_avalanche_pgf(pile, z)` evaluates $G$ at one complex $z$ by one tensor-train linear solve, and the distribution follows from $M$ evaluations on the unit circle by a discrete Fourier sum. The inversion aliases the tail: the value at $s$ carries the mass at $s + M$, $s + 2M$ and so on, so `SMAX` controls the aliasing error through $M = 2\,\mathrm{SMAX} + 2$. On this instance $\mathrm{E}[S]$ is of order 10.

In [6]:
pile = MannaSandpile(SITES, GRAINS)
dist_tail = exact_avalanche_size_distribution(pile, SMAX_TAIL)
mean_exact = float(np.sum(np.arange(SMAX_TAIL + 1) * dist_tail))
alias_bound = float(np.sum(dist_tail[2 * SMAX + 2:]))
print(f"ring of {SITES} sites, {GRAINS} grains; exact mean avalanche size {mean_exact:.12g}")
print(f"tail mass beyond the inversion window, the aliasing bound: {alias_bound:.1e}")

ring of 4 sites, 3 grains; exact mean avalanche size 3
tail mass beyond the inversion window, the aliasing bound: 3.0e-08


### Convergence of the bond dimension

The bond dimension `maxdim` caps the rank of every tensor-train solve, trading accuracy against cost, so it is a claim needing evidence, not a setting. This cell computes the mean avalanche size across a ladder of bond dimensions and takes the production value from where the answer stops moving; the exact mean above is the independent reference each rung is compared against. Every tensor-train number below this cell uses the value set here.

This ladder found a defect in an earlier version of the package, and the record is kept because a superseded result and its cause are part of the record. The assembly then compressed the operator itself under the solution's `maxdim`; at `maxdim = 8`, below this operator's middle-bond rank of 15, the solve satisfied a corrupted generator to a residual of order 1e-15 while its mean was wrong by half the exact value, because the residual certifies the solve of the system it was given, not the system's fidelity to the physics. The fix, applied to the Julia and Python implementations in the same change and pinned by a regression test in both suites, compresses operators and right-hand sides by the cutoff alone and reserves `maxdim` for solution tensors. The ladder below runs against the fixed code: where the cap is too small for the solution the residual is now large and says so, and where the cap suffices the mean lands on the exact value.

In [7]:
ladder = []
for maxdim in MAXDIM_LADDER:
    mean_tt, residual = tt_mean_avalanche_size(pile, cutoff=CUTOFF, maxdim=maxdim)
    ladder.append((maxdim, mean_tt, residual, abs(mean_tt - mean_exact)))
    print(f"maxdim {maxdim:3d}   mean {mean_tt:.12g}   residual {residual:.1e}   |mean - exact| {ladder[-1][3]:.1e}")

steps = [abs(ladder[i][1] - ladder[i - 1][1]) for i in range(1, len(ladder))]
converged = [MAXDIM_LADDER[i + 1] for i, s in enumerate(steps) if s < CONV_TOL]
assert converged, "the ladder never converged; extend MAXDIM_LADDER"
MAXDIM = converged[0]
print(f"production bond dimension MAXDIM = {MAXDIM}, first rung whose step is below {CONV_TOL:.0e}")

maxdim   4   mean 4.01320132013   residual 6.0e-01   |mean - exact| 1.0e+00
maxdim   8   mean 3   residual 1.2e-14   |mean - exact| 1.0e-13
maxdim  16   mean 3   residual 1.2e-14   |mean - exact| 1.0e-13
maxdim  32   mean 3   residual 1.2e-14   |mean - exact| 1.0e-13
production bond dimension MAXDIM = 16, first rung whose step is below 1e-09


### The avalanche-size distribution from every engine

The comparison is fair by construction, and this cell shows it rather than asserting it: the same `pile` instance, the same observable, the same `SMAX` window for the exact reference and the inversion, and the bond dimension fixed by the convergence cell above. The Monte Carlo histogram uses the same seed and shot count as the survival curve, and its regime differs in one stated way: with `NSHOTS` trajectories it cannot resolve probabilities much below 1/`NSHOTS`, so the tensor-train and exact tails continue below where its histogram ends. That difference is the finding the figure carries, not a defect of the comparison.

In [8]:
sizes = np.arange(SMAX + 1)
dist_exact = exact_avalanche_size_distribution(pile, SMAX)
dist_tt = tt_avalanche_size_distribution(pile, SMAX, cutoff=CUTOFF, maxdim=MAXDIM)
samples = kmc_avalanche_sizes(pile, NSHOTS, SEED)
hist = np.bincount(samples, minlength=SMAX + 1)[: SMAX + 1] / len(samples)

fig, ax = plt.subplots(figsize=(6.0, 3.6))
ax.plot(sizes, dist_exact, "o", ms=4, color=COLOUR_EXACT, label="exact")
ax.plot(sizes, hist, "s", ms=4, color=COLOUR_KMC, label=f"kmc, {NSHOTS} trajectories")
ax.plot(sizes, dist_tt, "^", ms=4, color=COLOUR_TT, label="tensor train")
ax.axhline(1.0 / NSHOTS, color=PLOT_DEFAULTS["guide_colour"], ls=":", lw=1.0)
ax.text(0.5, 1.3 / NSHOTS, "one count in NSHOTS", color=PLOT_DEFAULTS["guide_colour"], fontsize=9)
ax.set_xlabel("avalanche size S")
ax.set_ylabel("P(S)")
ax.set_yscale("log")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
deviation = max_abs_deviation(dist_tt, dist_exact)
print(f"largest tensor-train deviation from the exact distribution {deviation:.1e}")
print(f"aliasing bound from the long exact run {alias_bound:.1e}")
assert deviation < 10 * max(alias_bound, 1e-12), "inversion deviates beyond its stated aliasing bound"

largest tensor-train deviation from the exact distribution 1.5e-08
aliasing bound from the long exact run 3.0e-08


## Summary and caveats

The walk's hitting time landed on the gambler's ruin closed form from both the dense engine and the tensor-train solve, the survival curve from all three engines sits on the closed spectral form, and the sandpile's avalanche-size distribution from the Fourier-inverted generating function matches the exact embedded chain to its stated aliasing bound while resolving tail probabilities the Monte Carlo histogram does not reach. Each reported number was produced by a cell above and checked against an independent reference in the same cell.

Two caveats bound the conclusions. The aliasing bound depends on `SMAX`: shrink the window and the inversion folds real tail mass back onto small sizes, which the long exact run quantifies here but which nothing quantifies on instances too large for the exact engine. And the bond dimension was converged on this small ring; near the critical density of the conserved class the demanded rank grows, which is why the committed `manna_critical` instance runs Monte Carlo alone, the boundary the README's note on cost states. Everything downstream of these tables, classification, fitting, publication figures, is the reader's own analysis ([docs/notes.md](../docs/notes.md)).